# SumIterative, RecursiveSum and RecursiveSumLinear (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_python.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `SumIterative`, `RecursiveSum` and `RecursiveSumLinear`: the algorithms in Python, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursive_sum` (snake_case).

## The sum of an array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: a loop that accumulates the partial sums. $S(n)=3n+4$, extra memory $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide and conquer on the two halves (base case $l=r$). $2n-1$ calls, $S(n)=7n-5$, depth $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decrease by one: $S(n)=3n-1$, depth $n$.

*Input*: an array of $n\ge1$ integers (for `RecursiveSum`, two indices $l\le r$). *Output*: the sum of the elements. The counters `steps` (lines of pseudo-code executed), `calls` and `maxDepth` (simultaneously active calls) measure the quantities analysed in the chapter; in `RecursiveSum` and `RecursiveSumLinear` there is a single `return` so that the depth can be updated.

On the array of the example $a=[9,8,12,3,5,14,6]$ ($n=7$) the sum is $57$; `RecursiveSum` makes $13$ calls and $44$ steps, with depth $4$. Then we run the trace of the table of the chapter on $a=[9,8,12,3,5]$: the column *stack* reports the active calls.

In [1]:
steps = 0       # steps executed (lines of pseudo-code)
calls = 0       # calls executed
depth = 0       # calls active right now
max_depth = 0   # maximum depth of the recursion

def reset():
    global steps, calls, depth, max_depth
    steps = calls = depth = max_depth = 0

def sum_iterative(a, n):
    """Iterative sum of a[0..n-1]."""
    global steps
    s = 0
    i = 0
    steps += 2                  # lines 1-2: s := 0, i := 1
    while i < n:
        steps += 3              # true test + two lines of the body
        s = s + a[i]
        i = i + 1
    steps += 2                  # false test + return
    return s

def recursive_sum(a, l, r):
    """Sum of a[l..r] (0-based indices), l <= r, by divide and conquer."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # line 1: if l = r
    if l == r:
        steps += 1              # line 2: return a[l] (base case)
        result = a[l]
    else:
        steps += 4              # lines 3-6: m, two calls, return
        m = l + (r - l) // 2    # divide
        s1 = recursive_sum(a, l, m)         # conquer (left half)
        s2 = recursive_sum(a, m + 1, r)     # conquer (right half)
        result = s1 + s2                    # combine
    depth -= 1
    return result

def recursive_sum_linear(a, n):
    """Sum of a[0..n-1] by decrease by one, n >= 1."""
    global steps, calls, depth, max_depth
    calls += 1
    depth += 1
    max_depth = max(max_depth, depth)
    steps += 1                  # line 1: if n = 1
    if n == 1:
        steps += 1              # line 2: return a[1]
        result = a[0]
    else:
        steps += 2              # lines 3-4: call and return
        s = recursive_sum_linear(a, n - 1)
        result = s + a[n - 1]
    depth -= 1
    return result

# ---- example of the chapter ----
a = [9, 8, 12, 3, 5, 14, 6]
n = len(a)
print("a =", a, "  n =", n)
reset(); s = sum_iterative(a, n)
print(f"SumIterative       = {s}   steps={steps} (3n+4={3 * n + 4})")
reset(); s = recursive_sum(a, 0, n - 1)
print(f"RecursiveSum       = {s}   calls={calls} (2n-1={2 * n - 1})  steps={steps} (7n-5={7 * n - 5})  maxDepth={max_depth}")
reset(); s = recursive_sum_linear(a, n)
print(f"RecursiveSumLinear = {s}   calls={calls}  steps={steps} (3n-1={3 * n - 1})  maxDepth={max_depth}")

# ---- trace of RecursiveSum ----
row = 0
max_stack = 0

def emit(stack, action):
    global row, max_stack
    row += 1
    max_stack = max(max_stack, len(stack))
    print(f"{row:>2d} | {','.join(stack)} | {action}")

def recursive_sum_trace(a, l, r, stack):
    stack.append(f"({l},{r})")
    if l == r:
        emit(stack, f"l=r => return a[{l}]={a[l]}")
        stack.pop()
        return a[l]
    m = l + (r - l) // 2
    emit(stack, f"l!=r, m={l}+({r}-{l})/2={m}; call ({l},{m})")
    s1 = recursive_sum_trace(a, l, m, stack)
    emit(stack, f"s1={s1}; call ({m + 1},{r})")
    s2 = recursive_sum_trace(a, m + 1, r, stack)
    emit(stack, f"s2={s2}; return {s1}+{s2}={s1 + s2}")
    stack.pop()
    return s1 + s2

b = [9, 8, 12, 3, 5]
print()
print("a =", b, "  RecursiveSum(a, 0, 4)")
print(" # | stack | action")
res = recursive_sum_trace(b, 0, len(b) - 1, [])
print(f"result {res}, rows {row}, maximum number of active calls {max_stack}")

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | action
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Try it yourself.** Change the array `a` (also its length) and compare steps, calls and depth: only `RecursiveSumLinear` has depth $n$.

In [2]:
# Change the array: compare steps, calls and depth of the three algorithms.
a = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
n = len(a)
reset(); print(sum_iterative(a, n), steps)
reset(); print(recursive_sum(a, 0, n - 1), steps, calls, max_depth)
reset(); print(recursive_sum_linear(a, n), steps, calls, max_depth)

136 52
136 107 31 5
136 47 16 16


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).